In [ ]:
import yfinance as yf
from pandas import Timestamp
from tqdm.notebook import tqdm
import pandas as pd
import os
import numpy as np
import seaborn as sns


In [ ]:
seg = "Monthly"
level = "NFT_Level"
awareness = "directed/unaware"
k = 100
min_sim = 0.0
path = f"../graphs_(15-11)/{seg}/{level}/{awareness}/Top k {k} Min_sim {min_sim}"
SOGLIA = .0
SOGLIA_CORRELAZIONE = .0


In [ ]:
print("Time segmentation=", seg)
print("Graph Level=", level)
print("NFT awareness=", awareness)
print("Top k=", k)
print("Min sim=", min_sim)
print("SOGLIA", SOGLIA)


In [ ]:
def getTickers(tickers):
    series = None
    for ticker in tqdm(tickers):
        tk = yf.Ticker(ticker)
        if seg == "Weekly":
            prices = tk.history(period="4y", interval="1wk")
            prices["trimester"] = [
                str((Timestamp(t).year, (Timestamp(t).week-1))) for t in prices.index.values]
        if "Monthly" in seg:
            prices = tk.history(period="5y", interval="1mo")
            prices["trimester"] = [
                str((Timestamp(t).year, (Timestamp(t).month-1))) for t in prices.index.values]
        if seg == "Trimester":
            prices = tk.history(period="5y", interval="3mo")
            prices["trimester"] = [
                str((Timestamp(t).year, (Timestamp(t).month-1)//3)) for t in prices.index.values]

        prices.index = prices["trimester"]
        prices[ticker] = prices["Close"]
        if series is None:
            series = prices[[ticker]]
        else:
            series = series.join(prices[ticker])

    return series.drop_duplicates()


tickers = ["BTC-USD", "ETH-USD", "GAS-ETH"]
prices = getTickers(tickers)
prices["GAS-USD"] = prices["GAS-ETH"]*prices["ETH-USD"]
tickers = list(prices.columns.values)
s = sorted(os.listdir(path), key=lambda x: eval(x)[0]*100+eval(x)[1])


In [ ]:
def NftLevel_2_CollLevel(edge_list, info_list):
    colls = list(set(info_list.id.map(lambda x: eval(x)[0])))
    snapshots2idx = {s: i for i, s in enumerate(sorted(
        list(set(info_list.seen.values)), key=lambda x: eval(x)[0]*52+eval(x)[1]))}
    new_edgelist = edge_list.copy()
    new_edgelist["source_n"] = new_edgelist["source"]
    new_edgelist["target_n"] = new_edgelist["target"]
    info_list.index = info_list.id
    info_list_d = info_list.to_dict()
    new_edgelist["src_seen"] = new_edgelist["source"].apply(
        lambda x: snapshots2idx[info_list_d["seen"][x]])
    new_edgelist["trg_seen"] = new_edgelist["target"].apply(
        lambda x: snapshots2idx[info_list_d["seen"][x]])
    new_edgelist["source"] = new_edgelist["source"].map(lambda x: eval(x)[0])
    new_edgelist["target"] = new_edgelist["target"].map(lambda x: eval(x)[0])
    REMAPPED_EDGELIST = []
    for c1 in colls:
        c1_df = new_edgelist[new_edgelist.source == c1]
        for c2 in colls:
            if c1 != c2:
                c2_df = c1_df[c1_df.target == c2]

                c2_df=c2_df[c2_df["src_seen"]>c2_df["trg_seen"].min()]  # seleziono tutte e sole le similarità tali per cui src è nata prima della nascita di trg 

                
                REMAPPED_EDGELIST.append({
                        "source": c1,
                        "target": c2,
                        "weight": c2_df.weight.mean(),
                        "comparison_size": len(c2_df),
                        "src_seen_min": c2_df["src_seen"].min(),
                        "src_seen_max": c2_df["src_seen"].max(),
                        "trg_seen_min": c2_df["trg_seen"].min(),
                        "trg_seen_max": c2_df["trg_seen"].max(),
                    })

    REMAPPED_INFO = []
    for coll in colls:
        coll_info = info_list[info_list.collection == coll]
        REMAPPED_INFO.append({
            "id": coll,
            # prezzo medio di vendita della collection
            "avg": coll_info["avg"].mean(),
            # prezzo massimo di vendita della collection
            "max": coll_info["max"].max(),
            # prezzo minimo di vendita della collection
            "min": coll_info["min"].min(),
            # std medio di vendita della collection
            "std": coll_info["std"].mean(),
            # volume medio di vendita per ogni nft di questa collection
            "avgVolume": coll_info["volume"].mean(),
            # volume totale di vendita di questa collection
            "volume": coll_info["volume"].sum(),
            # numero totale di transazioni avvenute
            "#tx": coll_info["#tx"].sum(),
            "collection": coll_info["collection"].values[0],
            "category": coll_info["category"].values[0]
        })

    edge_list = pd.DataFrame().from_records(REMAPPED_EDGELIST)
    if len(edge_list)>0:
        edge_list=edge_list[edge_list.comparison_size>0]
    info_list = pd.DataFrame().from_records(REMAPPED_INFO, index="id")

    return edge_list, info_list


In [ ]:
EDGES_LIST = {}
INFO_LIST = {}
for snap in tqdm(s):
    info_list = pd.read_csv(f"{path}/{snap}/all_history_info.csv")
    try:
        edge_list = pd.read_csv(f"{path}/{snap}/edge_list.csv")
    except:
        edge_list = pd.DataFrame(columns=["source", "target", "weight","comparison_size"])
    edge_list, info_list = NftLevel_2_CollLevel(edge_list, info_list)
    INFO_LIST[snap] = info_list    
    os.makedirs(f"../graphs_(15-11)/{seg}/COLL_Level/directed/unaware_generated/Top k {k} Min_sim {min_sim}/{snap}",exist_ok=True)
    if len(edge_list)>0:
        edge_list[["source","target","weight"]].to_csv(f"../graphs_(15-11)/{seg}/COLL_Level/directed/unaware_generated/Top k {k} Min_sim {min_sim}/{snap}/edge_list.csv",index=False)
    else:
        edge_list.to_csv(f"../graphs_(15-11)/{seg}/COLL_Level/directed/unaware_generated/Top k {k} Min_sim {min_sim}/{snap}/edge_list.csv",index=False)

    INFO_LIST[snap].to_csv(f"../graphs_(15-11)/{seg}/COLL_Level/directed/unaware_generated/Top k {k} Min_sim {min_sim}/{snap}/all_history_info.csv")


In [ ]:
x = []
for snap in tqdm(s):

    edge_list, info_list = EDGES_LIST[snap], INFO_LIST[snap]

    x.append({
        "trimester": snap,
        "sum(volume)":              info_list["volume"].sum(),
        "mean(max(volume))":        info_list["max"].mean(),
        "mean(min(volume))":        info_list["min"].mean(),
        "mean(mean(volume))":       info_list["avg"].mean(),
        "sum(#tx)":                 info_list["#tx"].sum(),
        "max(#tx)":                 info_list["#tx"].max(),
        "min(#tx)":                 info_list["#tx"].min(),
        "mean(#tx)":                info_list["#tx"].mean(),
    })
volumes = pd.DataFrame.from_records(x)
volumes.index = volumes.trimester


In [ ]:
sim = []
BORN_NFT = {}
BORN_COLL = {}
for snap in tqdm(s):
    try:
        edge_list, info_list = EDGES_LIST[snap], INFO_LIST[snap]

        edges = edge_list[edge_list.weight > SOGLIA]
        info_list = info_list.to_dict()
        edges["category_src"] = edges["source"].apply(
            lambda x: info_list["category"][x])
        edges["category_dst"] = edges["target"].apply(
            lambda x: info_list["category"][x])

        sim.append({
            "trimester": snap,
            "sum(sim)": edges["weight"].sum(),
            "mean(sim)": edges["weight"].mean(),
            "intra": edges[edges.category_src == edges.category_dst].weight.mean(),
            "inter": edges[edges.category_src != edges.category_dst].weight.mean(),
            "intra_scaled": (edges[edges.category_src == edges.category_dst].weight*edges[edges.category_src == edges.category_dst].comparison_size).mean(),
            "inter_scaled": (edges[edges.category_src != edges.category_dst].weight*edges[edges.category_src != edges.category_dst].comparison_size).mean(),
            "comparison_size": (edges.comparison_size).mean(),
        })

    except Exception as e:
        print("Ex", e)
        sim.append({"trimester": snap,
                    "sum(sim)": 0,
                    "mean(sim)": 0,
                    "intra": 0,
                    "inter": 0,
                    })
sim = pd.DataFrame().from_records(sim)
sim.index = sim.trimester


In [ ]:
keys = set(prices.index.values)
keys.intersection_update(volumes.index.values)

keys.intersection_update(sim.index.values)

keys = list(set(keys))

keys = sorted(keys, key=lambda x: eval(x)[0]*52+eval(x)[1])
# BORN_COLL={k:BORN_COLL[k] for k in keys}
# BORN_NFT={k:BORN_NFT[k] for k in keys}

df = pd.concat([volumes.loc[keys], sim.loc[keys], prices.loc[keys]], axis=1)[
    list(volumes.columns.values)[1:]+list(sim.columns.values)[1:]+tickers]


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(4, 1, figsize=(21, 15))
df[["comparison_size", "inter_scaled", "intra_scaled"]].plot(ax=ax[0])
df[["inter", "intra", "mean(sim)"]].plot(ax=ax[1])
df[["mean(mean(volume))", "mean(min(volume))", "mean(max(volume))"]
   ].plot(ax=ax[2], logy=True)
df[tickers].plot(ax=ax[3], logy=True)


In [ ]:
plt.figure(figsize=(14, 12))
df_corr = df.corr()
df_corr = df_corr.where(np.abs(df_corr) > SOGLIA_CORRELAZIONE, 0)
sns.heatmap(df_corr, vmin=-1, vmax=1, annot=True,
            cmap=sns.color_palette("coolwarm", as_cmap=True))
print("Matrice di correlazione tra le grandezze osservate")
df_corr


In [ ]:
x, y = [], []
data = {}
for offset in range(1, 12):
    x.append(-offset)
    df_ = {
        f"GAS-ETH(past)":               df["GAS-ETH"][:-offset],
        f"ETH-USD(past)":               df["ETH-USD"][:-offset],
        f"BTC-USD(past)":               df["BTC-USD"][:-offset],
        f"sum(volume (past))":          df["sum(volume)"][:-offset],
        f"mean(mean(volume(past)))":          df["mean(mean(volume))"][:-offset],
        f"mean(max(volume(past)))":           df["mean(max(volume))"][:-offset],
        f"mean(min(volume(past)))":           df["mean(min(volume))"][:-offset],
        f"sum(#tx(past))":              df["sum(#tx)"][:-offset],
        f"mean(#tx(past))":             df["mean(#tx)"][:-offset],
        f"max(#tx(past))":              df["max(#tx)"][:-offset],
        f"min(#tx(past))":              df["min(#tx)"][:-offset],
        f"sum(sim(present))":           df["sum(sim)"][offset:],
        f"inter(present)":              df["inter"][offset:],
        f"intra(present)":              df["intra"][offset:],
        f"scaled_sim(present)":         df["intra"][offset:]/df["inter"][offset:],
        f"mean(sim(present))":          df["mean(sim)"][offset:],
    }
    df_ = pd.DataFrame().from_dict(df_)
    df_corr = df_.corr()
    df_corr = df_corr.where(np.abs(df_corr) > SOGLIA_CORRELAZIONE, 0)
    y.append(df_corr)
    data[-offset] = df_corr
fig, axes = plt.subplots(2, 3, figsize=(20, 10), sharex=True)
axes = axes.flatten()
fig.suptitle("Correlazioni per Mercati(riga 1) e Volume(riga 2) rispetto a similarità media, intra-category e inter-category")
ax = axes[0]
ax.plot(x, [d["inter(present)"][f"ETH-USD(past)"]
        for offset, d in enumerate(y, 1)], label="ETH(past)")
ax.plot(x, [d["inter(present)"][f"BTC-USD(past)"]
        for offset, d in enumerate(y, 1)], label="BTC(past)")
ax.plot(x, [d["inter(present)"][f"GAS-ETH(past)"]
        for offset, d in enumerate(y, 1)], label="GAS(past)")
ax.plot(x, [0 for _ in x], label="No corr")
ax.legend()
ax.set_title("inter-Category sim wrt market")

ax = axes[1]
ax.plot(x, [d["intra(present)"][f"ETH-USD(past)"]
        for offset, d in enumerate(y, 1)], label="ETH(past)")
ax.plot(x, [d["intra(present)"][f"BTC-USD(past)"]
        for offset, d in enumerate(y, 1)], label="BTC(past)")
ax.plot(x, [d["intra(present)"][f"GAS-ETH(past)"]
        for offset, d in enumerate(y, 1)], label="GAS(past)")
ax.plot(x, [0 for _ in x], label="No corr")
ax.legend()
ax.set_title("intra-Category sim wrt market")

ax = axes[2]
ax.plot(x, [d["scaled_sim(present)"][f"ETH-USD(past)"]
        for offset, d in enumerate(y, 1)], label="ETH(past)")
ax.plot(x, [d["scaled_sim(present)"][f"BTC-USD(past)"]
        for offset, d in enumerate(y, 1)], label="BTC(past)")
ax.plot(x, [d["scaled_sim(present)"][f"GAS-ETH(past)"]
        for offset, d in enumerate(y, 1)], label="GAS(past)")
ax.plot(x, [0 for _ in x], label="No corr")
ax.legend()
ax.set_title("scaled_sim(present) wrt market")

ax = axes[3]


ax.plot(x, [d[f"mean(max(volume(past)))"]["inter(present)"]
        for offset, d in enumerate(y, 1)], label="inter (present)")
ax.plot(x, [d[f"mean(max(volume(past)))"]["intra(present)"]
        for offset, d in enumerate(y, 1)], label="intra (present)")
ax.plot(x, [d[f"mean(max(volume(past)))"]["mean(sim(present))"]
        for offset, d in enumerate(y, 1)], label="mean(sim(present))")
ax.plot(x, [0 for _ in x], label="No corr")
ax.set_title("mean(max(volume)) wrt similarities")

ax.legend()

ax = axes[4]
ax.plot(x, [d[f"mean(min(volume(past)))"]["inter(present)"]
        for offset, d in enumerate(y, 1)], label="inter (present)")
ax.plot(x, [d[f"mean(min(volume(past)))"]["intra(present)"]
        for offset, d in enumerate(y, 1)], label="intra (present)")
ax.plot(x, [d[f"mean(min(volume(past)))"]["mean(sim(present))"]
        for offset, d in enumerate(y, 1)], label="mean(sim(present))")
ax.plot(x, [0 for _ in x], label="No corr")
ax.legend()
ax.set_title("min(volume) wrt similarities")

ax = axes[5]
ax.plot(x, [d[f"mean(mean(volume(past)))"]["inter(present)"]
        for offset, d in enumerate(y, 1)], label="inter (present)")
ax.plot(x, [d[f"mean(mean(volume(past)))"]["intra(present)"]
        for offset, d in enumerate(y, 1)], label="intra (present)")
ax.plot(x, [d[f"mean(mean(volume(past)))"]["mean(sim(present))"]
        for offset, d in enumerate(y, 1)], label="mean(sim(present))")
ax.plot(x, [0 for _ in x], label="No corr")
ax.legend()
ax.set_title("mean(volume)")

plt.show()


In [ ]:
from IPython.display import display
plt.figure(figsize=(10, 8))


def sortbyVolume(x):
    s = -(
        x[1]["mean(max(volume(past)))"]["mean(sim(present))"]
        # +x[1]["min(volume(past))"]["mean(sim(present))"]
        # +x[1]["mean(volume(past))"]["mean(sim(present))"]
    )
    return s


def sortbyTx(x):
    s = -(
        x[1]["max(#tx(past))"]["mean(sim(present))"] +
        x[1]["min(#tx(past))"]["mean(sim(present))"] +
        x[1]["mean(#tx(past))"]["mean(sim(present))"]
    )
    return s


offset = [(k, v) for k, v in sorted(
    data.items(), key=lambda x: sortbyVolume(x))][0][0]
# offset=-6
print(f"Offset mostrato {offset}")
df_ = data[offset]
df_ = df_[[k for k in df_.index.values if f'(past' not in k]].loc[[
    k for k in df_.index.values if f'(past' in k]]
df_ = df_.where(np.abs(df_) > SOGLIA_CORRELAZIONE, 0)
sns.heatmap(df_, vmin=-1, vmax=1, annot=True,
            cmap=sns.color_palette("coolwarm", as_cmap=True))
plt.show()
df_
